# MXnorm + ComBat + UniFORM Benchmark

Normalizes the PRAD-CyCIF dataset with independent baseline methods and evaluates
using the same metrics as `spancy_shift_explore.ipynb` — enabling direct benchmark comparison
against SpaNCy.

**Methods:**
- **MXnorm** (`mxnorm_registration`) — B-spline functional data registration
  (Harris et al., JOSS 2022). R package called via `rpy2`.
- **ComBat** (`combat`) — empirical Bayes batch correction
  (Johnson et al., Biostatistics 2007). Pure Python via `scanpy.pp.combat()`.
- **Z-Score** (`zscore`) — per-batch Z-score in log10 space, rescaled to global distribution.
- **UniFORM** (`uniform`) — histogram registration with GMM-based automatic landmark alignment
  (Wang et al., 2025). Pure Python via the UniFORM package.

These are **completely independent** implementations.

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2a. MXnorm Normalization (R)
2b. ComBat Normalization (Python)
2c. Z-Score Normalization (Python)
2d. UniFORM Normalization (Python)
3. Visual Inspection
4. Batch adj-R² Diagnostics
5. Positive Population Preservation Check
6. Histogram Comparison (PDF)
7. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs Python deps, R's mxnorm (for MXnorm only), and
uploads `spancy_shift.py` (shared utility functions for evaluation metrics).

In [ ]:
# System dep required by mxnorm compilation
!apt-get install -y cmake > /dev/null 2>&1

# Python dependencies
!pip install -q anndata scanpy pegasuspy pegasusio rpy2

# R packages for MXnorm (ComBat uses scanpy — not R)
!Rscript -e "if (!requireNamespace('BiocManager', quietly=TRUE)) install.packages('BiocManager', repos='https://cran.r-project.org')"
!Rscript -e "BiocManager::install('sva', ask=FALSE, update=FALSE)"
!Rscript -e "install.packages(c('mxnorm', 'fda'), repos='https://cran.r-project.org', quiet=TRUE)"

print('Done.')

In [ ]:
# Enable rpy2 magic cells (needed for MXnorm only)
%load_ext rpy2.ipython

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

print('rpy2 loaded, pandas2ri active')

In [ ]:
# Upload spancy_shift.py  (provides load_adata, per_marker_batch_r2, positive_population_table)
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found — upload it first'
print(f'spancy_shift.py uploaded ({os.path.getsize("spancy_shift.py"):,} bytes)')

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scanpy as sc

from spancy_shift import (
    load_adata,
    log1p_scale,
    detect_bimodal_markers,
    per_marker_batch_r2,
    positive_population_table,
)

# After re-uploading spancy_shift.py, run:
# import importlib; importlib.reload(spancy_shift); from spancy_shift import *

print('Imports OK')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

In [ ]:
# Install UniFORM — histogram registration method (Wang et al., 2025)
# Clones the repo and installs Python requirements (~1-2 min on fresh runtime)
!git clone https://github.com/kunlunW/UniFORM.git 2>/dev/null || echo 'UniFORM already cloned'
!pip install -q -r /content/UniFORM/requirements.txt

print('UniFORM ready.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f'\nMarkers ({adata.n_vars}): {list(adata.var_names)}')
print(f'Batches:  {sorted(adata.obs["batch_id"].unique().tolist())}')
print(f'\nobs columns: {list(adata.obs.columns)}')

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)

In [ ]:
batch_vals     = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].boxplot(
    [X_raw[:, i] for i in range(X_raw.shape[1])],
    labels=marker_names, vert=True
)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')

for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2a. MXnorm Normalization (R)

**MXnorm** (Harris et al., JOSS 2022) normalizes multiplexed imaging data using
B-spline functional data registration. It is an R package called here via rpy2.

**Input**: `expm1(adata.X)` — raw linear values (MXnorm applies `transform="log"` internally,
so results come back in the same log1p-equivalent space as all other methods).

Reference: https://github.com/ColemanRHarris/mxnorm

In [ ]:
# X_raw is in raw linear space (raw CyCIF intensities).
# Pre-transform to log10(x+1) space ourselves so we control the +1 offset (handles zeros).
# Pass to MXnorm with transform="None" — Registration works on our log10 values.

X_raw_log10 = np.log10(X_raw.astype(np.float64) + 1.0)

df_mxnorm = pd.DataFrame(X_raw_log10, columns=marker_names)
df_mxnorm['batch_id']  = adata.obs['batch_id'].values
df_mxnorm['sample_id'] = adata.obs['sample_id'].values

marker_names_r = marker_names  # passed to R as character vector

print(f'DataFrame shape:      {df_mxnorm.shape}')
print(f'log10(x+1) range:     [{X_raw_log10.min():.4f}, {X_raw_log10.max():.4f}]')
print(f'Inf/NaN check:        {np.isinf(X_raw_log10).sum()} Inf, {np.isnan(X_raw_log10).sum()} NaN')
print(f'Batches: {sorted(df_mxnorm["batch_id"].unique().tolist())}')

In [ ]:
%%R -i df_mxnorm -i marker_names_r -o result_reg_df
library(mxnorm)

marker_names_r <- as.character(unlist(marker_names_r))

mx_data <- mx_dataset(
  data        = df_mxnorm,
  slide_id    = "batch_id",
  image_id    = "sample_id",
  marker_cols = marker_names_r
)
cat(sprintf("mx_dataset: %d cells, %d markers, %d batches\n",
            nrow(df_mxnorm), length(marker_names_r),
            length(unique(df_mxnorm$batch_id))))

# transform="None": data is already log10(x+1) from Python, no additional transform needed
cat("Running MXnorm Registration...\n")
mx_reg <- mx_normalize(mx_data, transform="None", method="Registration")
result_reg_df <- mx_reg$norm_data[, marker_names_r, drop=FALSE]
cat(sprintf("Done. Range: [%.4f, %.4f]\n", min(result_reg_df), max(result_reg_df)))

In [ ]:
# MXnorm output is in log10(x+1) space (same as input — transform="None" was used).
# Inverse: 10^x - 1 to convert back to linear space matching X_raw.
X_mxnorm_log10 = result_reg_df[marker_names].values.astype(np.float64)
X_mxnorm = np.power(10.0, X_mxnorm_log10) - 1.0
X_mxnorm = np.clip(X_mxnorm, 0, None).astype(np.float32)

adata.layers['mxnorm_registration'] = X_mxnorm

L = adata.layers['mxnorm_registration']
print(f'mxnorm_registration  shape={L.shape}  '
      f'range=[{L.min():.4f}, {L.max():.4e}]  '
      f'NaN={np.isnan(L).sum()}  Inf={np.isinf(L).sum()}')

## 2b. ComBat Normalization (Python)

**ComBat** (Johnson et al., Biostatistics 2007) removes batch effects via empirical Bayes.
Implemented here as `scanpy.pp.combat()` — pure Python, fully independent of MXnorm and R.

**Input**: `adata.X` (log1p space) — ComBat was designed for log-scale expression data.

In [ ]:
# ComBat expects log-scale data. X_raw is linear, so apply log1p first,
# run ComBat, then convert back to linear with expm1.
adata_tmp = adata.copy()

X_log1p = np.log1p(
    np.asarray(adata_tmp.X.toarray() if sp.issparse(adata_tmp.X) else adata_tmp.X,
               dtype=np.float64)
)
adata_tmp.X = X_log1p

sc.pp.combat(adata_tmp, key='batch_id')

X_combat = np.expm1(
    np.asarray(adata_tmp.X, dtype=np.float64)
)
X_combat = np.clip(X_combat, 0, None).astype(np.float32)

adata.layers['combat'] = X_combat
del adata_tmp

L = adata.layers['combat']
print(f'combat               shape={L.shape}  '
      f'range=[{L.min():.4f}, {L.max():.4e}]  '
      f'NaN={np.isnan(L).sum()}  Inf={np.isinf(L).sum()}')

print(f'\nAll layers: {list(adata.layers.keys())}')

## 2c. Z-Score Normalization (Python)

Per-batch z-score in log10(x+1) space, rescaled back to the global distribution.
For each marker and each batch: subtract batch mean, divide by batch std, then rescale
to global mean and std. Converts back to linear space with `10^x − 1`.

This is a simple statistical baseline — no model, no R required.

In [ ]:
X_log10 = np.log10(X_raw.astype(np.float64) + 1.0)

batch_labels = adata.obs['batch_id'].values
unique_b     = np.unique(batch_labels)

global_mean = X_log10.mean(axis=0)
global_std  = X_log10.std(axis=0)
global_std  = np.where(global_std < 1e-8, 1.0, global_std)

X_zs = X_log10.copy()
for b in unique_b:
    mask       = batch_labels == b
    b_mean     = X_log10[mask].mean(axis=0)
    b_std      = X_log10[mask].std(axis=0)
    b_std      = np.where(b_std < 1e-8, 1.0, b_std)
    # z-score within batch, rescale to global distribution
    X_zs[mask] = (X_log10[mask] - b_mean) / b_std * global_std + global_mean

X_zscore = np.power(10.0, X_zs) - 1.0
X_zscore = np.clip(X_zscore, 0, None).astype(np.float32)

adata.layers['zscore'] = X_zscore

L = adata.layers['zscore']
print(f'zscore               shape={L.shape}  '
      f'range=[{L.min():.4f}, {L.max():.4e}]  '
      f'NaN={np.isnan(L).sum()}  Inf={np.isinf(L).sum()}')
print(f'\nAll layers: {list(adata.layers.keys())}')

## 2d. UniFORM Normalization (Python)

**UniFORM** (Wang et al., 2025) normalizes multiplexed imaging data using
histogram registration with GMM-based automatic landmark detection.
Pure Python — no R required.

Normalization applies a **per-sample per-marker multiplicative scale factor** derived from
histogram alignment in log-space. Input and output are both in raw linear space.

Reference: https://github.com/kunlunW/UniFORM

In [ ]:
import sys
sys.path.insert(0, '/content')

from UniFORM.UniFORM.preprocessing import process_sample_distributions
from UniFORM.UniFORM.registration import automatic_registration
from UniFORM.UniFORM.normalization import generate_normalized_feature
import anndata as ad

# Minimal AnnData copy — avoids writing our other layers to Anndata_Normalized.h5ad
adata_uniform = ad.AnnData(
    X=X_raw.astype(np.float64),
    obs=adata.obs[['sample_id', 'batch_id']].copy(),
    var=adata.var.copy(),
)
adata_uniform.obs['sample_id'] = adata_uniform.obs['sample_id'].astype(str)

# Step 1: compute intensity ranges, histograms, GMM models per sample
intensity_ranges, histograms, gmm_models = process_sample_distributions(
    feature_input      = adata_uniform,
    num_bins           = 1024,
    plots_per_row      = 4,
    dpi                = 100,
    xlims              = None,
    ylims              = None,
    output_figure_path = "uniform_feature_distributions.png",
    verbose            = False,
)

# Step 2: automatic landmark registration (no manual picking)
shifts_map, chosen_references, implied_landmarks_map = automatic_registration(
    histogram_data    = histograms,
    all_markers       = adata_uniform.var['marker_name'].tolist(),
    selected_markers  = adata_uniform.var['marker_name'].tolist(),
    sample_ids        = adata_uniform.obs['sample_id'].unique().tolist(),
    reference_samples = None,
    landmark_map      = None,
    num_bins          = 1024,
    dpi               = 100,
    x_limits          = None,
)

# Step 3: apply normalization — stores result in adata_uniform.layers['normalized']
generate_normalized_feature(
    feature_input            = adata_uniform,
    sample_ids               = None,
    markers                  = None,
    intensity_ranges         = intensity_ranges,
    shifts_map               = shifts_map,
    chosen_references        = chosen_references,
    num_bins                 = 1024,
    dpi                      = 100,
    plot_dist                = False,
    plot_single_cell_corr    = False,
    gmm_analysis             = False,
    save_normalized_features = True,
)

# Copy to main adata as 'uniform' layer (same linear space as X_raw)
X_uniform = np.asarray(adata_uniform.layers['normalized'], dtype=np.float32)
X_uniform = np.clip(X_uniform, 0, None)
adata.layers['uniform'] = X_uniform
del adata_uniform

L = adata.layers['uniform']
print(f'uniform              shape={L.shape}  '
      f'range=[{L.min():.4f}, {L.max():.4e}]  '
      f'NaN={np.isnan(L).sum()}  Inf={np.isinf(L).sum()}')
print(f'\nAll layers: {list(adata.layers.keys())}')

## 3. Visual Inspection

Bimodal marker classification (reused in Sections 5–6) and
distribution comparison across all methods.

In [ ]:
X_scaled, _  = log1p_scale(X_raw)
batch_codes  = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_scaled, marker_names,
    batch_codes=batch_codes,
    min_prominence_frac=0.02,
    bimodal_min_batch_frac=0.5,
)

bimodal_names  = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]
unimodal_names = [marker_names[i] for i in range(len(marker_names)) if not marker_is_bimodal[i]]

print(f'Bimodal ({len(bimodal_names)}):  {bimodal_names}')
print(f'Unimodal ({len(unimodal_names)}): {unimodal_names}')

In [ ]:
plot_markers = bimodal_names[:4] if len(bimodal_names) >= 4 else \
               bimodal_names + marker_names[:max(0, 4 - len(bimodal_names))]

layers_to_show = [
    (None,                   'Raw'),
    ('mxnorm_registration',  'MXnorm Registration'),
    ('combat',               'ComBat'),
    ('zscore',               'Z-Score'),
    ('uniform',              'UniFORM'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        data = X_raw if layer_key is None else adata.layers[layer_key]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle('Bimodal Marker Distributions: Raw → MXnorm → ComBat → Z-Score → UniFORM', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels (one-hot).
Lower = batch effect better removed.  Target < 0.05 per marker.

In [ ]:
batch_arr = adata.obs['batch_id'].values

r2_raw     = per_marker_batch_r2(X_raw,                                 batch_arr, marker_names)
r2_mxnorm  = per_marker_batch_r2(adata.layers['mxnorm_registration'],   batch_arr, marker_names)
r2_combat  = per_marker_batch_r2(adata.layers['combat'],                batch_arr, marker_names)
r2_zscore  = per_marker_batch_r2(adata.layers['zscore'],                batch_arr, marker_names)
r2_uniform = per_marker_batch_r2(adata.layers['uniform'],               batch_arr, marker_names)

r2_compare = (
    r2_raw.rename(columns={'adj_r2': 'raw'})
    .merge(r2_mxnorm.rename(columns={'adj_r2': 'mxnorm_registration'}), on='marker')
    .merge(r2_combat.rename(columns={'adj_r2': 'combat'}), on='marker')
    .merge(r2_zscore.rename(columns={'adj_r2': 'zscore'}), on='marker')
    .merge(r2_uniform.rename(columns={'adj_r2': 'uniform'}), on='marker')
)

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in ['raw', 'mxnorm_registration', 'combat', 'zscore', 'uniform']:
    print(f'  {col:>22s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))
x = np.arange(len(r2_compare))
w = 0.15
ax.bar(x - 2*w, r2_compare['raw'],                  w, label='Raw',                 color='salmon',      alpha=0.85)
ax.bar(x -   w, r2_compare['mxnorm_registration'],   w, label='MXnorm Registration', color='steelblue',   alpha=0.85)
ax.bar(x      , r2_compare['combat'],                w, label='ComBat',              color='forestgreen', alpha=0.85)
ax.bar(x +   w, r2_compare['zscore'],                w, label='Z-Score',             color='mediumpurple', alpha=0.85)
ax.bar(x + 2*w, r2_compare['uniform'],               w, label='UniFORM',             color='darkorange',  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Positive Population Preservation Check

% positive cells per marker per sample using GMM threshold (2-component Gaussian mixture; Otsu fallback).
Delta = normalized − raw.  Target: |delta| < 5% per marker.

In [ ]:
pop_table = positive_population_table(
    adata, norm_layer='mxnorm_registration', sample_col='sample_id'
)

summary = pop_table.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (mxnorm_registration vs raw):')
print('=' * 65)
print(summary.to_string())
print(f'\nOverall mean |delta|: {pop_table["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table["delta"].abs().max():.2f}%')

In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: MXnorm Registration vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
plot_markers_pop = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names[:2]

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table[pop_table['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (MXnorm Registration)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation — MXnorm Registration', fontsize=13)
plt.tight_layout()
plt.show()

### 5b. Positive Population Preservation — ComBat

In [ ]:
pop_table_combat = positive_population_table(
    adata, norm_layer='combat', sample_col='sample_id'
)

summary_combat = pop_table_combat.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (combat vs raw):')
print('=' * 65)
print(summary_combat.to_string())
print(f'\nOverall mean |delta|: {pop_table_combat["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table_combat["delta"].abs().max():.2f}%')

In [ ]:
pivot_combat = pop_table_combat.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_combat.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot_combat.columns)))
ax.set_xticklabels(pivot_combat.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_combat.index)))
ax.set_yticklabels(pivot_combat.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: ComBat vs Raw')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table_combat[pop_table_combat['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (ComBat)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation — ComBat', fontsize=13)
plt.tight_layout()
plt.show()

### 5c. Positive Population Preservation — Z-Score

In [ ]:
pop_table_zscore = positive_population_table(
    adata, norm_layer='zscore', sample_col='sample_id'
)

summary_zscore = pop_table_zscore.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (zscore vs raw):')
print('=' * 65)
print(summary_zscore.to_string())
print(f'\nOverall mean |delta|: {pop_table_zscore["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table_zscore["delta"].abs().max():.2f}%')

pivot_zscore = pop_table_zscore.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_zscore.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot_zscore.columns)))
ax.set_xticklabels(pivot_zscore.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_zscore.index)))
ax.set_yticklabels(pivot_zscore.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: Z-Score vs Raw')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table_zscore[pop_table_zscore['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (Z-Score)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation — Z-Score', fontsize=13)
plt.tight_layout()
plt.show()

### 5d. Positive Population Preservation — UniFORM

In [ ]:
pop_table_uniform = positive_population_table(
    adata, norm_layer='uniform', sample_col='sample_id'
)

summary_uniform = pop_table_uniform.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (uniform vs raw):')
print('=' * 65)
print(summary_uniform.to_string())
print(f'\nOverall mean |delta|: {pop_table_uniform["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table_uniform["delta"].abs().max():.2f}%')

pivot_uniform = pop_table_uniform.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_uniform.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot_uniform.columns)))
ax.set_xticklabels(pivot_uniform.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_uniform.index)))
ax.set_yticklabels(pivot_uniform.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: UniFORM vs Raw')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table_uniform[pop_table_uniform['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (UniFORM)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation — UniFORM', fontsize=13)
plt.tight_layout()
plt.show()

### 5e. Positive Population Change — All Methods (Bubble Plot)

Bubble size ∝ |mean Δ|, color = inter-sample SD (blue = consistent, red = variable).
2×2 grid: one panel per method. Shared colorbar and ±5% target zone.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np, pandas as pd

METHODS_DICT = {
    'MXnorm':  pop_table,
    'ComBat':  pop_table_combat,
    'Z-Score': pop_table_zscore,
    'UniFORM': pop_table_uniform,
}

# --- Aggregate mean and SD of delta per marker ---
agg = {}
for method_name, pt in METHODS_DICT.items():
    g = pt.groupby('marker')['delta'].agg(['mean', 'std']).reset_index()
    g.columns = ['marker', 'mean_delta', 'std_delta']
    agg[method_name] = g

order        = list(adata.var_names)
method_names = list(METHODS_DICT.keys())
n_methods    = len(method_names)
n_markers    = len(order)

all_stds   = pd.concat([g['std_delta']        for g in agg.values()])
all_abs    = pd.concat([g['mean_delta'].abs() for g in agg.values()])
vmin, vmax = 0.0, float(all_stds.max())
size_scale = 800.0 / max(float(all_abs.max()), 1.0)
cmap       = plt.cm.coolwarm

fig, ax = plt.subplots(figsize=(2 + n_methods * 2, max(8, n_markers * 0.45)))

for x_idx, method_name in enumerate(method_names):
    g     = agg[method_name].set_index('marker').loc[order].reset_index()
    y_pos = np.arange(n_markers)
    sizes = np.abs(g['mean_delta'].values) * size_scale + 20
    ax.scatter(
        np.full(n_markers, x_idx), y_pos,
        s=sizes, c=g['std_delta'],
        cmap=cmap, vmin=vmin, vmax=vmax,
        edgecolors='k', linewidths=0.4, alpha=0.88,
    )

ax.set_xticks(range(n_methods))
ax.set_xticklabels(method_names, fontsize=11)
ax.set_yticks(range(n_markers))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlim(-0.6, n_methods - 0.4)
ax.set_ylim(-0.7, n_markers - 0.3)
ax.invert_yaxis()
ax.grid(True, alpha=0.2)

for idx, mname in enumerate(order):
    if mname in bimodal_names:
        ax.get_yticklabels()[idx].set_color('navy')
        ax.get_yticklabels()[idx].set_fontweight('bold')

for ref_delta in [5, 10, 20]:
    ax.scatter([], [], s=ref_delta * size_scale + 20, c='gray', alpha=0.7,
               edgecolors='k', linewidths=0.4, label=f'{ref_delta}%')
ax.legend(title='|mean Δ|', loc='lower right', fontsize=8, title_fontsize=8, framealpha=0.9)

plt.colorbar(
    cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap),
    ax=ax,
    label='SD of Δ across samples\n(blue = consistent  ·  red = variable)',
    shrink=0.6,
)
ax.set_title(
    'Positive Population Change — All Benchmark Methods\n'
    'Bubble size ∝ |mean Δ|  ·  Color = inter-sample SD  ·  Target: |Δ| < 5%\n'
    'Navy bold = bimodal markers',
    fontsize=11,
)
plt.tight_layout()
plt.show()

# --- Summary table ---
frames = []
for method_name, pt in METHODS_DICT.items():
    tmp = pt[['marker', 'delta']].copy()
    tmp['method'] = method_name
    frames.append(tmp)
all_delta = pd.concat(frames, ignore_index=True)

pivot = (
    all_delta.groupby(['marker', 'method'])['delta']
    .agg(['mean', 'std']).round(2)
    .unstack('method')
)
print('\nMean Δ (±SD) positive population per marker:')
print('=' * 70)
print(pivot.to_string())
print('\nTarget: |mean Δ| < 5% per marker')

## 6. Histogram Comparison

Per-sample histogram PDFs for all markers.  Saved to `histograms_mxnorm/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_mxnorm'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('mxnorm_registration',  'MXnorm Registration'),
    ('combat',               'ComBat'),
    ('zscore',               'Z-Score'),
    ('uniform',              'UniFORM'),
]

sample_col     = 'sample_id' if 'sample_id' in adata.obs.columns else 'batch_id'
sample_ids     = adata.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap   = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path      = os.path.join(save_dir, 'mxnorm_histograms.pdf')
n_cols        = len(layers_to_plot)
rows_per_page = 4

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), n_cols,
                                     figsize=(7 * n_cols, 4 * len(page_markers)))
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)
            if n_cols == 1:
                axes = axes.reshape(-1, 1)

            for row, mname in enumerate(page_markers):
                m_idx = marker_names.index(mname)
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_col = np.array(adata.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, e = np.histogram(X_log[mask], bins=80)
                        ax.plot(e[:-1], c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')

## 7. kBET Evaluation

Compute kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- **Lower chi-square = better**

Group definitions and pegasus pipeline are **identical** to `spancy_shift_explore.ipynb`.

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['mxnorm_registration', 'combat', 'zscore', 'uniform']

print(f'Evaluating layers: {EVAL_LAYERS}')

In [ ]:
all_kbet = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata.copy()
    adata_kbet.X = adata.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts   = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata  = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            if np.isnan(score):
                raise ValueError('kBET returned NaN')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 80)
print('kBET Detailed Results — per clinical group')
print('=' * 80)

# Build per-group table
rows = []
for layer_name, res in all_kbet.items():
    for gname, vals in res.items():
        rows.append({
            'layer': layer_name,
            'group': gname,
            'kBET':  vals['kBET'],
            'chi2':  vals['chi2'],
            'p':     vals['p'],
        })

df_detail = pd.DataFrame(rows)
print(df_detail.to_string(index=False, float_format='%.4f'))

# Mean summary
print('\n' + '=' * 80)
print('kBET Summary (mean across groups — higher kBET = better, lower chi2 = better, higher p = better)')
print('=' * 80)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    pvals = [v['p']    for v in res.values() if not np.isnan(v['p'])]
    if kbets:
        row = {
            'layer':     layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'mean_p':    float(np.mean(pvals)),
            'n_groups':  f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>25s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  p={row['mean_p']:.4f}  ({row['n_groups']} groups)")

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color='steelblue', alpha=0.85)
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color='salmon', alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    axes[2].bar(df_kbet['layer'], df_kbet['mean_p'], color='mediumseagreen', alpha=0.85)
    axes[2].set_ylabel('Mean p-value')
    axes[2].set_title('p-value (higher = better)')
    axes[2].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()